# Mygrate Interactive Runner
This notebook sets up the environment and exposes a single interactive cell to invoke the Mygrate orchestrator (LLM-driven) or a deterministic pipeline.
- Imports and setup are done automatically.
- Provide a `project_path` and whether to use LLM.
- Visualizations are optional and will be generated if `migration_intelligence.json` is present.

In [ ]:
import os
import sys
import json
import subprocess
import importlib.util
from pathlib import Path
from IPython.display import display, Image, Markdown, clear_output

# Ensure repo root on path
sys.path.append(os.path.abspath('..')) if os.path.basename(os.getcwd()) == 'notebooks' else sys.path.append(os.path.abspath('.'))

def check_javac():
    try:
        p = subprocess.run(['javac', '-version'], capture_output=True, text=True)
        out = p.stderr.strip() or p.stdout.strip()
        return (p.returncode == 0), out
    except FileNotFoundError:
        return False, 'javac not found in PATH'

# Try to import workflow.app (orchestrator). If unavailable we'll detect and fallback.
try:
    from src.workflow import app
    HAVE_WORKFLOW = True
    _WORKFLOW_ERR = ''
except Exception as e:
    HAVE_WORKFLOW = False
    _WORKFLOW_ERR = str(e)

def run_with_workflow(project_path: str, instruction: str, thread_id: str = 'mygrate_interactive'):
    if not HAVE_WORKFLOW:
        return False, f'workflow unavailable: {_WORKFLOW_ERR}'
    config = {'{'}'configurable': {'{'}'thread_id': thread_id{'}'}{'}'}
    initial_state = {
        'project_path': project_path,
        'target_java_version': '17',
        'messages': [],
        'completed_tasks_summary': [],
        'dependencies': [],
        'compatibility_matrix': {},
        'migration_tasks': [],
        'current_instruction': instruction,
        'last_subagent_result': '',
        'next_node': 'supervisor'
    }
    try:
        app.invoke(initial_state, config=config)
        # auto-run routed subagents until end
        while True:
            state_val = app.get_state(config).values
            next_node = state_val.get('next_node', 'end')
            if next_node == 'end':
                break
            app.invoke(None, config=config)
        latest = app.get_state(config).values
        return True, latest.get('last_subagent_result', '')
    except Exception as e:
        return False, str(e)

def run_local_pipeline(project_path: str):
    # Load and call scripts/run_local_pipeline.py::run(project_path) dynamically
    script_path = os.path.join(os.getcwd(), 'scripts', 'run_local_pipeline.py')
    if not os.path.exists(script_path):
        return 1, {'error': 'run_local_pipeline.py not found'}
    spec = importlib.util.spec_from_file_location('run_local_pipeline', script_path)
    module = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(module)
        rc = module.run(project_path)
    except Exception as e:
        return 1, {'error': str(e)}
    # attempt to read migration_intelligence.json
    intel = {}
    try:
        with open('migration_intelligence.json', 'r', encoding='utf-8') as f:
            intel = json.load(f)
    except Exception:
        pass
    return rc, intel

def visualize_if_present(intel_path: str = 'migration_intelligence.json', ask_user: bool = True):
    if not os.path.exists(intel_path):
        print(f'No intelligence file found at {intel_path}')
        return
    if ask_user:
        ans = input('Generate visualizations now? [y/N]: ').strip().lower()
        if ans not in ('y','yes'):
            print('Skipping visualizations.')
            return
    try:
        from src.tools.visualization_engine import generate_dashboard, generate_cross_matrix
        generate_dashboard(intel_path, 'dependency_graph.png')
        generate_cross_matrix(intel_path, 'cross_matrix.png')
        display(Markdown('**Generated:** dependency_graph.png, cross_matrix.png'))
        if os.path.exists('dependency_graph.png'):
            display(Image('dependency_graph.png'))
        if os.path.exists('cross_matrix.png'):
            display(Image('cross_matrix.png'))
    except Exception as e:
        print('Visualization error:', e)

print('Setup complete. Use the next cell to run the pipeline interactively.')

In [ ]:
# Interactive runner cell
project = input('Project path (relative to repo root) [freshbrew_data/cantor]: ').strip() or 'freshbrew_data/cantor'
use_llm = input('Use LLM/orchestrator if available? [y/N]: ').strip().lower() in ('y','yes')
visualize = input('Prompt for visualizations after run? [Y/n]: ').strip().lower() not in ('n','no')
instruction = input('Instruction for agents (e.g.
): ').strip() or 'Quét dự án và sinh ma trận tương thích'

print('
[+] Checking javac...')
javac_ok, javac_msg = check_javac()
print('javac:', javac_ok, javac_msg)

result = None
if use_llm and HAVE_WORKFLOW:
    print('[+] Attempting LLM-driven orchestrator (Supervisor)...')
    ok, payload = run_with_workflow(project, instruction)
    if ok:
        print('[+] Orchestrator completed. Agent output (last_subagent_result):')
        print(payload[:1000])
        result = payload
    else:
        print('[!] Orchestrator failed or returned no result. Falling back to deterministic pipeline.')
        rc, intel = run_local_pipeline(project)
        print(f'[+] Deterministic pipeline returned code {rc}')
        print(json.dumps(intel, indent=2)[:2000])
        result = intel
else:
    print('[+] Running deterministic pipeline (tools-only)')
    rc, intel = run_local_pipeline(project)
    print(f'[+] Deterministic pipeline returned code {rc}')
    print(json.dumps(intel, indent=2)[:2000])
    result = intel

# Optional visualizations
if visualize:
    visualize_if_present('migration_intelligence.json', ask_user=True)

# Expose `result` in notebook namespace for further inspection
_mygrate_last_result = result
print('
[+] Done. The variable `_mygrate_last_result` contains the returned object (or string).')